In [1]:
import pandas as pd
from datetime import datetime
import sqlite3
import random
from sqlalchemy import create_engine, text
import os

In [4]:
DB_HOST = os.getenv("DB_HOST", "127.0.0.1")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "vendor_db")
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "postgres")

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
conn = create_engine(DATABASE_URL)

In [3]:
# conn = sqlite3.connect("./my_database.db")

In [5]:
vendor_ids = pd.read_sql("select distinct(vendor_id) from ticket", con=conn)

In [6]:
test_vendor = vendor_ids.iloc[2]['vendor_id']
test_vendor

'ac1f8f53-4a58-4534-8c62-a45c9e873b7d'

In [8]:
df = pd.read_sql("select * from ticket where invoice_id is not null", con=conn)
df.head()

,ticket_id,work_order_id,invoice_id,description,assigned_contractor,priority,status,vendor_id,start_time,due_date,...,estimated_quantity,unit,special_requirements,anomaly_flag,anomaly_reason,created_at,updated_at,created_by,updated_by,additional_information
0,64dab536-fb1d-41fb-bbba-2d51c7f2d667,0afe97cf-0e47-4fd5-840d-e32c6a83e447,14c40a8d-3a6c-46c6-b44b-6a4bbf99822f,Action movie choose because school energy. Uni...,84806496-bb33-419b-b028-85e5bda25b27,MEDIUM,COMPLETED,477ff30f-7e84-459e-9fb8-5793ff5a4fa2,2025-09-23 05:22:21.770996+00:00,2025-09-26 20:13:36.605417+00:00,...,47.39,hours,High example role body simply forget represent.,False,NaN,2025-09-22 05:43:18.295813+00:00,2026-04-14 02:25:34.335893+00:00,a067f877-bcad-47c1-ba25-7ccb90c8196d,3bf6c279-da9b-442f-8ad2-d254da53e31d,"{'source': 'sample_generator', 'has_photo': Fa..."
1,09f0662f-29c3-45f2-a384-f5e26d20914a,2844433c-2985-4166-9297-5f3997bdbf41,af2ff1b2-ba9a-4eee-a8df-f9611434bc2f,Statement less open just military. Heart by ti...,95aa70a0-bfda-4691-a2fc-f2f04d6a78e4,HIGH,COMPLETED,cf76ad77-20a2-4e52-8585-fa7a8459e6e4,2025-02-06 18:02:28.968727+00:00,2025-02-09 20:22:29.070113+00:00,...,78.78,visits,Put yourself religious cause design year.,True,Certainly over bring police really.,2025-02-03 18:59:07.813502+00:00,2026-04-14 02:25:34.336580+00:00,8a472195-e246-493a-af27-b58f9dddd9fc,5da77ab2-8fa3-4343-a70b-3d966df03bcd,"{'source': 'sample_generator', 'has_photo': Fa..."
2,9995d8d1-ceb0-4d53-b55c-ae3585f6c544,829b028f-daa9-4d49-b83f-49bb17698a05,9d871d9c-3eb8-466b-8681-50afe8d95880,Meeting expert claim life either mother instea...,fb391fd8-07da-4c03-a9de-ff5933592ca6,HIGH,COMPLETED,7c6b28e0-93a8-4a8f-9653-5673e107b09a,2024-08-23 05:18:26.201526+00:00,2024-08-25 08:13:11.522041+00:00,...,62.95,hours,Fill she network base.,True,NaN,2024-08-19 14:07:46.465841+00:00,2026-04-14 02:25:34.338519+00:00,4e1d5b5c-01d9-48a1-907e-1deabc550fc5,70ea70f2-12f0-4693-9364-a5f0c1c465b1,"{'source': 'sample_generator', 'has_photo': Fa..."
3,3a1c8930-7c3a-4332-a998-f4635a62b044,b95b2f77-f14f-4f2c-9a58-8eed18772fdb,e26fdc2f-f03f-48d0-a8ae-aeb453abcda3,Network word consumer life visit success set. ...,11fd3f1c-55b5-4ca3-8445-d0f8d033610d,LOW,COMPLETED,bac9a3aa-d44d-4177-88a4-86c75973e665,2024-08-05 20:17:52.645108+00:00,2024-08-07 00:06:04.468784+00:00,...,31.21,loads,Staff design sport opportunity suggest.,False,Soon eight behind fact these.,2024-08-04 00:05:40.512534+00:00,2026-04-14 02:25:34.339152+00:00,af79c052-b030-4691-9e8c-65aae11316dc,9f0488cd-3649-4e4d-9cdf-7b2114ce6e03,"{'source': 'sample_generator', 'has_photo': Fa..."
4,9a3a19db-b1a2-46c7-a1ca-acb1790a817c,1b98e86d-afef-4627-8442-edfca99a0f57,1c4997e0-1be5-4c6c-923e-a4d0b9ee9bc3,Usually central Congress can middle bag stay. ...,98d9c117-fac2-4f0c-8129-84b65a25bc07,MEDIUM,COMPLETED,c203322e-f39b-444b-98f7-965b3070cf9c,2024-10-06 19:26:23.986603+00:00,2024-10-11 16:54:01.036199+00:00,...,74.96,loads,No spring out start reason floor.,False,NaN,2024-10-06 17:27:36.071277+00:00,2026-04-14 02:25:34.340965+00:00,aed0f485-a014-4e53-88e4-1a0241b69ef9,f3193abc-bc35-497f-947c-8942caad8e46,"{'source': 'sample_generator', 'has_photo': Fa..."


In [9]:
def unassigned_work_orders(vendor_id):
    # df = pd.read_sql(
    #     f"""
    #     select count(*) as unassigned_work_orders
    #     from work_orders
    #     where assigned_vendor='{vendor_id}'"""
    # , con=conn)
    df = pd.read_sql(
        f"""
        select work_order_id, count(*) as "unassigned tickets"
        from ticket t
        join work_orders wo
        using (work_order_id)
        where t.status = 'UNASSIGNED' and wo.assigned_vendor = '{vendor_id}'
        group by 1;
        """, con=conn
    )
    return {'vendor_id': vendor_id, 'unassigned tickets': df['unassigned tickets'].sum()}

In [11]:
unassigned_work_orders(test_vendor)

{'vendor_id': '87557121-007b-4e66-9a6e-733b6e2bc363',
 'unassigned tickets': np.int64(94)}

In [12]:
def pending_tickets(vendor_id):
    df = pd.read_sql(f"""
    select count(*) as active_tickets
    from ticket 
    where vendor_id='{vendor_id}' and (status = 'IN_PROGRESS' or status = 'ASSIGNED')
    """, con=conn)
    count = df['active_tickets'].iloc[0]
    return {'vendor_id': vendor_id, 'active tickets': count}

In [13]:
pending_tickets(test_vendor)

{'vendor_id': '87557121-007b-4e66-9a6e-733b6e2bc363',
 'active tickets': np.int64(217)}

In [30]:
df = pd.read_sql(
    f"""select * 
    from ticket 
    where vendor_id='{test_vendor}' and completed_at is not null and created_at > NOW() - interval '12 months'"""
    , con=conn, parse_dates=['assigned_at', 'completed_at'])

df.head(1)

,ticket_id,work_order_id,invoice_id,description,assigned_contractor,priority,status,vendor_id,start_time,due_date,...,estimated_quantity,unit,special_requirements,anomaly_flag,anomaly_reason,created_at,updated_at,created_by,updated_by,additional_information
0,7f0efafc-b797-49ce-afde-1ef439453ed7,9dfc6b0e-94b6-42aa-b76d-c868a8760cb2,NaN,Late hand way network good some respond or. Co...,b560b26d-e5d4-489e-bcf3-88877d42cb92,LOW,COMPLETED,87557121-007b-4e66-9a6e-733b6e2bc363,2025-07-07 15:09:52.882270+00:00,2025-07-12 03:43:38.662657+00:00,...,82.49,hours,Size all eight large create left room.,False,NaN,2025-07-05 17:02:14.505882+00:00,2026-04-12 23:49:46.316706+00:00,8b8807f4-c988-4fd1-8838-a2d500359eb1,f078e66c-2df3-4350-940b-8e9d7dbf7bc7,"{'source': 'sample_generator', 'has_photo': True}"


In [31]:
df['completion_week'] = df['completed_at'].dt.to_period('W').apply(lambda r: r.end_time).dt.date

C:\Users\danie\AppData\Local\Temp\ipykernel_14896\1373384851.py:1: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['completion_week'] = df['completed_at'].dt.to_period('W').apply(lambda r: r.end_time).dt.date


In [32]:
complete_by_week = df.groupby('completion_week')['ticket_id'].count().reset_index().sort_values(by='ticket_id', ascending=False)
complete_by_week.rename(columns={'ticket_id': 'ticket_count'}, inplace=True)
complete_by_week

,completion_week,ticket_count
2,2025-07-13,16
3,2025-07-20,9
7,2026-02-01,6
4,2025-07-27,3
1,2025-05-11,2
0,2025-05-04,1
5,2025-09-07,1
6,2026-01-25,1


## Invoice KPIs

In [12]:
invoices = pd.read_sql(f"select * from invoice", con=conn)
invoices.head()

,invoice_id,work_order_id,vendor_id,client_id,invoice_date,due_date,period_start,period_end,total_amount,invoice_status,paid_at,approved_at,rejected_at,created_at,updated_at,created_by,updated_by
0,f7271ea5-9735-43dc-944e-7a03d3aff5c5,2165112f-b59d-4e4e-9bb1-f7bb2a8fe8a1,c203322e-f39b-444b-98f7-965b3070cf9c,1d650d85-fe37-4520-a427-b43a988bb33f,2025-12-15 18:58:02.993928+00:00,2026-01-12 01:12:50.269178+00:00,2025-12-04 03:30:49.249488+00:00,2025-12-09 17:27:24.867899+00:00,2811.89,REJECTED,NaT,NaT,2025-12-22 10:26:57.286174+00:00,2026-04-14 02:25:35.478913+00:00,2026-04-14 02:25:35.478915+00:00,667e1ad3-b40f-45d6-8605-93834bcdee02,76f4bdbf-1d82-49ce-ada8-d6cf3abc451f
1,1702cb34-8260-418a-83d1-12b032a315b3,7d84c53a-dd8e-4f46-ba32-d3a3893205a2,7c6b28e0-93a8-4a8f-9653-5673e107b09a,7b57e2ca-44f4-47b1-94da-90c7145dd487,2025-04-12 15:24:24.451766+00:00,2025-04-20 09:14:23.974071+00:00,2025-04-09 08:47:09.617446+00:00,2025-04-10 11:26:49.350374+00:00,12837.59,SUBMITTED,NaT,NaT,NaT,2026-04-14 02:25:35.478956+00:00,2026-04-14 02:25:35.478956+00:00,4119c0bd-7946-4bda-acc3-1c83f82add1d,41506358-22a1-45c3-ae10-8602fadec7b4
2,df6e4ba4-d908-42d2-bc2a-cdf7e000425f,25cc858f-84ca-40c9-a7fe-d2845d40e68f,2a2f5fda-4e07-45f9-aa97-7dbd69b7709e,dcdcf237-185c-4ee1-bb09-6346a696bc6f,2025-12-30 23:59:19.506764+00:00,2026-01-19 03:26:42.808099+00:00,2025-12-22 11:18:24.768549+00:00,2025-12-24 18:05:19.854614+00:00,5463.66,REJECTED,NaT,NaT,2026-01-02 17:51:04.639695+00:00,2026-04-14 02:25:35.479001+00:00,2026-04-14 02:25:35.479001+00:00,77e64db2-9b3a-4a76-801f-d0b1f07320cc,557b0caf-b0d2-493a-926a-1c6fed911f97
3,c1462d92-8223-4572-820c-bf20765ff686,3706fb04-6513-4f96-b23a-ac93180322ad,ac1f8f53-4a58-4534-8c62-a45c9e873b7d,351603aa-4240-49ce-b617-de27042d5136,2026-02-22 13:18:16.895525+00:00,2026-03-15 08:49:06.422191+00:00,2026-02-08 05:17:50.403432+00:00,2026-02-14 02:56:27.010275+00:00,2865.04,APPROVED,NaT,2026-03-01 07:07:54.522943+00:00,NaT,2026-04-14 02:25:35.479042+00:00,2026-04-14 02:25:35.479043+00:00,deb740a4-e432-4f15-8ec9-2c4221a43262,88255ebb-2856-48d3-a5ec-dc72c57e26e8
4,16b28829-ad9f-4c7f-814c-4c197117dc7c,a232a134-871f-4c28-91a6-d950ce1d9794,ac1f8f53-4a58-4534-8c62-a45c9e873b7d,4e71fdf0-05ff-433b-9591-4698d4d30f38,2024-05-18 12:19:09.702584+00:00,2024-06-06 08:21:29.904655+00:00,2024-05-11 09:34:06.638446+00:00,2024-05-11 19:33:55.448491+00:00,10653.71,DRAFT,NaT,NaT,NaT,2026-04-14 02:25:35.479073+00:00,2026-04-14 02:25:35.479073+00:00,982ff773-43f3-4665-b163-1600f5155c2d,3fe8bcb1-64a5-476e-9000-17ceeb31afe6


In [14]:
vendor_invoices = invoices[invoices['vendor_id'] == test_vendor]

In [17]:
invoices['invoice_status'].value_counts()

invoice_status
SUBMITTED    105
DRAFT        104
REJECTED     103
APPROVED      96
PAID          92
Name: count, dtype: int64

In [23]:
invoice_insights = vendor_invoices[vendor_invoices['invoice_status'].isin(['SUBMITTED', 'REJECTED', 'APPROVED'])]

In [24]:
invoice_insights.columns

Index(['invoice_id', 'work_order_id', 'vendor_id', 'client_id', 'invoice_date',
       'due_date', 'period_start', 'period_end', 'total_amount',
       'invoice_status', 'paid_at', 'approved_at', 'rejected_at', 'created_at',
       'updated_at', 'created_by', 'updated_by'],
      dtype='str')

In [25]:
invoice_insights.groupby('invoice_status')['total_amount'].sum()

invoice_status
APPROVED     69957.33
REJECTED     67565.72
SUBMITTED    31608.56
Name: total_amount, dtype: float64